# Data Processing

## Imports

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

from pathlib import Path

spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/21 22:48:45 WARN Utils: Your hostname, MacBook-Air-de-Vitor.local, resolves to a loopback address: 127.0.0.1; using 192.168.3.49 instead (on interface en0)
26/05/21 22:48:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/21 22:48:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Load

In [26]:
data_path = Path.cwd().parent / 'data' / 'raw'

profile_schema = StructType([
    StructField("age", LongType(), True),
    StructField("credit_card_limit", DoubleType(), True),
    StructField("gender", StringType(), True),
    StructField("id", StringType(), True),
    StructField("registered_on", DateType(), True)
])

offers = spark.read.json(str(data_path / 'offers.json'))
profile = (
    spark.read.json(str(data_path / 'profile.json'), schema=profile_schema, dateFormat="yyyyMMdd")
    .withColumn("age", F.when(F.col("age")!=118, F.col("age")))
)
transactions = (
    spark.read.json(str(data_path / 'transactions.json'))
    .withColumn("offer_id", F.coalesce(F.col("value.offer id"), F.col("value.offer_id")))
    .withColumn("transaction_amount", F.col("value.amount"))
    .withColumn("offer_reward", F.col("value.reward"))
    .drop("value")
)


# TODO
# 2. Remove age 118 e garantir que é 118 quando o resto é null
# Temos que criar um modelo para ajudar definir qual o melhor cliente para oferecer um desconto

## Validations

### Offers

In [4]:
offers.count()

10

In [5]:
offers.printSchema()

root
 |-- channels: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- discount_value: long (nullable = true)
 |-- duration: double (nullable = true)
 |-- id: string (nullable = true)
 |-- min_value: long (nullable = true)
 |-- offer_type: string (nullable = true)



In [6]:
offers.orderBy("offer_type", "channels").show(10, truncate=False)

+----------------------------+--------------+--------+--------------------------------+---------+-------------+
|channels                    |discount_value|duration|id                              |min_value|offer_type   |
+----------------------------+--------------+--------+--------------------------------+---------+-------------+
|[email, mobile, social]     |10            |7.0     |ae264e3637204a6fb9bb56bc8210ddfd|10       |bogo         |
|[web, email, mobile]        |5             |7.0     |9b98b8c7a33c4b65b9aebfe6a799e6d9|5        |bogo         |
|[web, email, mobile, social]|10            |5.0     |4d5c57ea9a6940dd891ad53e9dbe8da0|10       |bogo         |
|[web, email, mobile, social]|5             |5.0     |f19421c1d4aa40978ebb69ca19b0e20d|5        |bogo         |
|[web, email]                |5             |10.0    |0b1e1539f2cc45b7b9fa7c272da2e1d7|20       |discount     |
|[web, email, mobile]        |2             |7.0     |2906b810c7d4411798c6938adc9daaa5|10       |discoun

In [7]:
(
    offers
    .groupBy("channels")
    .count()
).show(truncate=False)

+----------------------------+-----+
|channels                    |count|
+----------------------------+-----+
|[web, email]                |1    |
|[email, mobile, social]     |2    |
|[web, email, mobile]        |3    |
|[web, email, mobile, social]|4    |
+----------------------------+-----+



### Profile

In [8]:
profile.count()

17000

In [9]:
profile.printSchema()

root
 |-- age: long (nullable = true)
 |-- credit_card_limit: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- id: string (nullable = true)
 |-- registered_on: date (nullable = true)



In [10]:
profile.show()

+---+-----------------+------+--------------------+-------------+
|age|credit_card_limit|gender|                  id|registered_on|
+---+-----------------+------+--------------------+-------------+
|118|             NULL|  NULL|68be06ca386d4c319...|   2017-02-12|
| 55|         112000.0|     F|0610b486422d4921a...|   2017-07-15|
|118|             NULL|  NULL|38fe809add3b4fcf9...|   2018-07-12|
| 75|         100000.0|     F|78afa995795e4d85b...|   2017-05-09|
|118|             NULL|  NULL|a03223e636434f42a...|   2017-08-04|
| 68|          70000.0|     M|e2127556f4f64592b...|   2018-04-26|
|118|             NULL|  NULL|8ec6ce2a7e7949b1b...|   2017-09-25|
|118|             NULL|  NULL|68617ca6246f4fbc8...|   2017-10-02|
| 65|          53000.0|     M|389bc3fa690240e79...|   2018-02-09|
|118|             NULL|  NULL|8974fc5686fe429db...|   2016-11-22|
|118|             NULL|  NULL|c4863c7985cf408fa...|   2017-08-24|
|118|             NULL|  NULL|148adfcaa27d485b8...|   2015-09-19|
| 58|     

In [11]:
profile.groupBy("age").count().orderBy("age").plot(kind='bar', x='age', y='count')

In [12]:
profile.agg(F.mean(F.col("age").isNull().cast("int"))).show()

+-------------------------------+
|avg(CAST((age IS NULL) AS INT))|
+-------------------------------+
|                            0.0|
+-------------------------------+



In [13]:
(
    profile
    .groupBy(
        F.col("age") == 118,
        F.col("credit_card_limit").isNull(),
        F.col("gender").isNull()
    )
    .count()
).show()

+-----------+---------------------------+----------------+-----+
|(age = 118)|(credit_card_limit IS NULL)|(gender IS NULL)|count|
+-----------+---------------------------+----------------+-----+
|      false|                      false|           false|14825|
|       true|                       true|            true| 2175|
+-----------+---------------------------+----------------+-----+



In [14]:
profile.groupby("gender").count().show()

+------+-----+
|gender|count|
+------+-----+
|     F| 6129|
|  NULL| 2175|
|     M| 8484|
|     O|  212|
+------+-----+



In [15]:
profile.select("credit_card_limit").describe().show()

+-------+------------------+
|summary| credit_card_limit|
+-------+------------------+
|  count|             14825|
|   mean|  65404.9915682968|
| stddev|21598.299410229436|
|    min|           30000.0|
|    max|          120000.0|
+-------+------------------+



### Transactions

In [16]:
transactions.count()

306534

In [17]:
transactions.printSchema()

root
 |-- account_id: string (nullable = true)
 |-- event: string (nullable = true)
 |-- time_since_test_start: double (nullable = true)
 |-- value: struct (nullable = true)
 |    |-- amount: double (nullable = true)
 |    |-- offer id: string (nullable = true)
 |    |-- offer_id: string (nullable = true)
 |    |-- reward: double (nullable = true)



In [ ]:
transaction_transaction = transactions.where(F.col("event")=="transaction")
transaction_viewed = transactions.where(F.col("event")=="offer viewed")
transaction_received = transactions.where(F.col("event")=="offer received")
transaction_completed = transactions.where(F.col("event")=="offer completed")

In [25]:
(
    transaction_completed
    .groupBy(
        F.col("value.amount").isNull(),
        F.col("value.offer id").isNull(),
        F.col("value.offer_id").isNull(),
        F.col("value.reward").isNull(),
    )
    .count()
).show()


+----------------------+------------------------+------------------------+----------------------+-----+
|(value.amount IS NULL)|(value.offer id IS NULL)|(value.offer_id IS NULL)|(value.reward IS NULL)|count|
+----------------------+------------------------+------------------------+----------------------+-----+
|                  true|                    true|                   false|                 false|33579|
+----------------------+------------------------+------------------------+----------------------+-----+



In [24]:
(
    transaction_received 
    .groupBy(
        F.col("value.amount").isNull(),
        F.col("value.offer id").isNull(),
        F.col("value.offer_id").isNull(),
        F.col("value.reward").isNull(),
    )
    .count()
).show()


+----------------------+------------------------+------------------------+----------------------+-----+
|(value.amount IS NULL)|(value.offer id IS NULL)|(value.offer_id IS NULL)|(value.reward IS NULL)|count|
+----------------------+------------------------+------------------------+----------------------+-----+
|                  true|                   false|                    true|                  true|76277|
+----------------------+------------------------+------------------------+----------------------+-----+



In [23]:
(
    transaction_viewed 
    .groupBy(
        F.col("value.amount").isNull(),
        F.col("value.offer id").isNull(),
        F.col("value.offer_id").isNull(),
        F.col("value.reward").isNull(),
    )
    .count()
).show()


+----------------------+------------------------+------------------------+----------------------+-----+
|(value.amount IS NULL)|(value.offer id IS NULL)|(value.offer_id IS NULL)|(value.reward IS NULL)|count|
+----------------------+------------------------+------------------------+----------------------+-----+
|                  true|                   false|                    true|                  true|57725|
+----------------------+------------------------+------------------------+----------------------+-----+



In [22]:
(
    transaction_transaction
    .groupBy(
        F.col("value.amount").isNull(),
        F.col("value.offer id").isNull(),
        F.col("value.offer_id").isNull(),
        F.col("value.reward").isNull(),
    )
    .count()
).show()


+----------------------+------------------------+------------------------+----------------------+------+
|(value.amount IS NULL)|(value.offer id IS NULL)|(value.offer_id IS NULL)|(value.reward IS NULL)| count|
+----------------------+------------------------+------------------------+----------------------+------+
|                 false|                    true|                    true|                  true|138953|
+----------------------+------------------------+------------------------+----------------------+------+



In [42]:
transactions.orderBy(F.rand()).show(truncate=False)

+--------------------------------+---------------+---------------------+----------------------------------------------------+
|account_id                      |event          |time_since_test_start|value                                               |
+--------------------------------+---------------+---------------------+----------------------------------------------------+
|a8dc1ad7fda243edad2b45982dd541bb|transaction    |24.25                |{14.26, NULL, NULL, NULL}                           |
|09a5c16b6b9d4aa8a6d1301620c0e900|transaction    |0.25                 |{0.29, NULL, NULL, NULL}                            |
|248dc37acf3f4197b0a92bda480e57a5|offer viewed   |17.25                |{NULL, 4d5c57ea9a6940dd891ad53e9dbe8da0, NULL, NULL}|
|f1f01be4e2344bf08b7f488c115ad374|offer viewed   |20.0                 |{NULL, 3f207df678b143eea3cee63160fa8bed, NULL, NULL}|
|13853a0e60ad42bea59e490fb39b0044|offer viewed   |23.75                |{NULL, 9b98b8c7a33c4b65b9aebfe6a799e6d9, NULL,

In [39]:
transactions.groupBy("event").count().show()

+---------------+------+
|          event| count|
+---------------+------+
|    transaction|138953|
| offer received| 76277|
|offer completed| 33579|
|   offer viewed| 57725|
+---------------+------+



In [40]:
transactions.where(F.col("time_since_test_start")==0).groupBy("event").count().show()

+---------------+-----+
|          event|count|
+---------------+-----+
|    transaction|  633|
| offer received|12650|
|offer completed|  206|
|   offer viewed| 2072|
+---------------+-----+

